<a href="https://colab.research.google.com/github/nitekar/DentalTest/blob/main/notebook/DentalTest_Improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dental Radiography ML Pipeline - Complete Implementation

This notebook implements a comprehensive deep learning pipeline for dental X-ray classification.

**Classes**: `BDC-BDR`, `Caries`, `Fractured Teeth`, `Healthy Teeth`, `Impacted teeth`, `Infection`

**Features**:
- Data preprocessing with CLAHE enhancement
- Transfer learning with ResNet18
- Data augmentation
- Model training and evaluation
- Performance metrics and visualization
- Model saving and deployment preparation

---

## 🚀 Quick Start

### Running on Google Colab:
1. Click the "Open in Colab" badge above
2. Run the first code cell to mount Google Drive and clone the repository
3. Run all cells sequentially

### Running Locally:
1. Ensure you're in the project root directory
2. Run all cells sequentially

---

## 1. Import Libraries and Setup

## 0. Setup for Google Colab

Run this cell if you're on Google Colab to mount Drive and install dependencies.

In [ ]:
# Check if running on Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running on Google Colab")
    
    # Mount Google Drive (optional, for saving models)
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Install required packages
    !pip install -q albumentations opencv-python-headless kagglehub
    
    # Option 1: Download dataset from Kaggle (RECOMMENDED)
    print("\nDownloading dataset from Kaggle...")
    import kagglehub
    dataset_path = kagglehub.dataset_download("imtkaggleteam/dental-radiography")
    print(f"Dataset downloaded to: {dataset_path}")
    
    # Option 2: Clone from GitHub (if you have the data in the repo)
    # !git clone https://github.com/nitekar/DentalTest.git
    # %cd DentalTest
    
    # Option 3: If data is in Google Drive
    # dataset_path = '/content/drive/MyDrive/DentalTest/data/Dental OPG (Classification)/'
    
else:
    print("Running locally")
    # Set current directory
    import os
    os.chdir('../..')
    dataset_path = None  # Will use default path
    print(f"Current directory: {os.getcwd()}")

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Deep Learning Libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models
import torch.nn.functional as F

# Sklearn for metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 2. Data Loading and Exploration

In [ ]:
# Define data paths
# Adjust path based on environment
if IN_COLAB:
    base_path = "data/Dental OPG (Classification)/"
else:
    base_path = "data/Dental OPG (Classification)/"

categories = ["BDC-BDR", "Caries", "Fractured Teeth", "Healthy Teeth", "Impacted teeth", "Infection"]

# Create dataset DataFrame
def create_dataset_df(base_path, categories):
    image_paths = []
    labels = []
    
    for category in categories:
        category_path = os.path.join(base_path, category)
        if os.path.exists(category_path):
            for image_name in os.listdir(category_path):
                if image_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    image_path = os.path.join(category_path, image_name)
                    image_paths.append(image_path)
                    labels.append(category)
    
    return pd.DataFrame({
        "image_path": image_paths,
        "label": labels
    })

# Create dataset
df = create_dataset_df(base_path, categories)
print(f"Total images: {len(df)}")
print(f"Dataset shape: {df.shape}")
print("\nClass distribution:")
print(df['label'].value_counts())

In [ ]:
# Visualize class distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Bar plot
df['label'].value_counts().plot(kind='bar', ax=ax1, color='skyblue')
ax1.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Classes')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=45)

# Add count labels on bars
for i, v in enumerate(df['label'].value_counts().values):
    ax1.text(i, v + 5, str(v), ha='center', va='bottom')

# Pie chart
df['label'].value_counts().plot(kind='pie', ax=ax2, autopct='%1.1f%%')
ax2.set_title('Class Distribution (Percentage)', fontsize=14, fontweight='bold')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

## 3. Image Preprocessing with CLAHE

In [ ]:
def apply_clahe(image_path, clip_limit=2.0, tile_grid_size=(8, 8)):
    """
    Apply CLAHE (Contrast Limited Adaptive Histogram Equalization) to enhance image contrast
    """
    # Read image
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    
    if image is None:
        return None
    
    # Create CLAHE object
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    
    # Apply CLAHE
    enhanced_image = clahe.apply(image)
    
    # Convert to RGB for consistency
    enhanced_image = cv2.cvtColor(enhanced_image, cv2.COLOR_GRAY2RGB)
    
    return enhanced_image

# Test CLAHE on sample images
def show_clahe_comparison(df, num_samples=3):
    fig, axes = plt.subplots(2, num_samples, figsize=(15, 8))
    
    sample_images = df.sample(num_samples)
    
    for i, (_, row) in enumerate(sample_images.iterrows()):
        # Original image
        original = cv2.imread(row['image_path'])
        original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
        
        # CLAHE enhanced image
        enhanced = apply_clahe(row['image_path'])
        
        if enhanced is not None:
            axes[0, i].imshow(original)
            axes[0, i].set_title(f'Original - {row["label"]}')
            axes[0, i].axis('off')
            
            axes[1, i].imshow(enhanced)
            axes[1, i].set_title(f'CLAHE Enhanced - {row["label"]}')
            axes[1, i].axis('off')
    
    plt.suptitle('CLAHE Enhancement Comparison', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_clahe_comparison(df)

## 4. Custom Dataset Class

In [ ]:
class DentalDataset(Dataset):
    def __init__(self, dataframe, transform=None, use_clahe=True):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.use_clahe = use_clahe
        
        # Create label encoder
        self.label_encoder = LabelEncoder()
        self.labels = self.label_encoder.fit_transform(dataframe['label'])
        self.class_names = self.label_encoder.classes_
        
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['image_path']
        label = self.labels[idx]
        
        # Load and preprocess image
        if self.use_clahe:
            image = apply_clahe(img_path)
        else:
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if image is None:
            # Return a black image if loading fails
            image = np.zeros((224, 224, 3), dtype=np.uint8)
        
        # Convert to PIL Image for transforms
        image = Image.fromarray(image)
        
        if self.transform:
            image = self.transform(image)
        
        return image, label
    
    def get_class_names(self):
        return self.class_names

## 5. Data Augmentation and Transforms

In [ ]:
# Define transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Split dataset
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"Train set: {len(train_df)} images")
print(f"Validation set: {len(val_df)} images")
print(f"Test set: {len(test_df)} images")

# Create datasets
train_dataset = DentalDataset(train_df, transform=train_transform)
val_dataset = DentalDataset(val_df, transform=val_transform)
test_dataset = DentalDataset(test_df, transform=val_transform)

# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print(f"\nClass names: {train_dataset.get_class_names()}")
print(f"Number of classes: {len(train_dataset.get_class_names())}")

## 6. Model Architecture (ResNet18 with Transfer Learning)

In [ ]:
class DentalClassifier(nn.Module):
    def __init__(self, num_classes=6, pretrained=True):
        super(DentalClassifier, self).__init__()
        
        # Load pretrained ResNet18
        self.backbone = models.resnet18(pretrained=pretrained)
        
        # Modify the final layer
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        return self.backbone(x)

# Initialize model
num_classes = len(train_dataset.get_class_names())
model = DentalClassifier(num_classes=num_classes).to(device)

# Print model summary
print(f"Model initialized with {num_classes} classes")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 7. Training Setup

In [ ]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

# Training function
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc

# Validation function
def validate_epoch(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            
            running_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc

## 8. Model Training

In [ ]:
# Training loop
num_epochs = 25
best_val_acc = 0.0
train_losses = []
train_accs = []
val_losses = []
val_accs = []

print("Starting training...")
print("-" * 60)

for epoch in range(num_epochs):
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
    
    # Update learning rate
    scheduler.step(val_loss)
    
    # Store metrics
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        os.makedirs('models', exist_ok=True)
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_acc': best_val_acc,
            'class_names': train_dataset.get_class_names()
        }, 'models/dental_model_best.pth')
    
    # Print progress
    print(f'Epoch [{epoch+1}/{num_epochs}]')
    print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
    print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
    print(f'Best Val Acc: {best_val_acc:.2f}%')
    print("-" * 60)

print(f"Training completed! Best validation accuracy: {best_val_acc:.2f}%")

## 9. Training Visualization

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
ax1.plot(train_losses, label='Training Loss', color='blue')
ax1.plot(val_losses, label='Validation Loss', color='red')
ax1.set_title('Model Loss', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# Accuracy plot
ax2.plot(train_accs, label='Training Accuracy', color='blue')
ax2.plot(val_accs, label='Validation Accuracy', color='red')
ax2.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# Print final metrics
print(f"Final Training Accuracy: {train_accs[-1]:.2f}%")
print(f"Final Validation Accuracy: {val_accs[-1]:.2f}%")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")

## 10. Model Evaluation

In [ ]:
# Load best model
checkpoint = torch.load('models/dental_model_best.pth')
model.load_state_dict(checkpoint['model_state_dict'])
class_names = checkpoint['class_names']

# Evaluate on test set
def evaluate_model(model, test_loader, device, class_names):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, predicted = torch.max(output, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(target.cpu().numpy())
    
    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    
    print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))
    
    return all_labels, all_preds

test_labels, test_preds = evaluate_model(model, test_loader, device, class_names)

## 11. Confusion Matrix Visualization

In [ ]:
# Plot confusion matrix
def plot_confusion_matrix(y_true, y_pred, class_names):
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    # Calculate per-class accuracy
    class_accuracy = cm.diagonal() / cm.sum(axis=1)
    
    print("\nPer-class Accuracy:")
    for i, acc in enumerate(class_accuracy):
        print(f"{class_names[i]}: {acc:.4f} ({acc*100:.2f}%)")

plot_confusion_matrix(test_labels, test_preds, class_names)

## 12. Sample Predictions Visualization

In [ ]:
def visualize_predictions(model, test_dataset, device, class_names, num_samples=12):
    model.eval()
    
    # Get random samples
    indices = np.random.choice(len(test_dataset), num_samples, replace=False)
    
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.ravel()
    
    with torch.no_grad():
        for i, idx in enumerate(indices):
            # Get image and true label
            image, true_label = test_dataset[idx]
            
            # Make prediction
            image_batch = image.unsqueeze(0).to(device)
            output = model(image_batch)
            probabilities = F.softmax(output, dim=1)
            confidence, predicted = torch.max(probabilities, 1)
            
            # Convert image for display
            img_display = image.permute(1, 2, 0)
            img_display = img_display * torch.tensor([0.229, 0.224, 0.225]) + torch.tensor([0.485, 0.456, 0.406])
            img_display = torch.clamp(img_display, 0, 1)
            
            # Plot
            axes[i].imshow(img_display)
            axes[i].axis('off')
            
            true_class = class_names[true_label]
            pred_class = class_names[predicted.item()]
            conf = confidence.item()
            
            color = 'green' if true_label == predicted.item() else 'red'
            axes[i].set_title(f'True: {true_class}\nPred: {pred_class}\nConf: {conf:.3f}', 
                             color=color, fontsize=10)
    
    plt.suptitle('Sample Predictions', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_predictions(model, test_dataset, device, class_names)

## 13. Model Saving for Deployment

In [ ]:
# Save final model for deployment
final_model_path = 'models/dental_model.pth'

# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Save complete model information
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': class_names,
    'num_classes': len(class_names),
    'model_architecture': 'ResNet18',
    'input_size': (224, 224),
    'best_val_acc': best_val_acc,
    'test_accuracy': accuracy_score(test_labels, test_preds),
    'normalization_params': {
        'mean': [0.485, 0.456, 0.406],
        'std': [0.229, 0.224, 0.225]
    }
}, final_model_path)

print(f"Model saved to: {final_model_path}")
print(f"Model size: {os.path.getsize(final_model_path) / (1024*1024):.2f} MB")

# Create model info summary
model_info = {
    'Model Architecture': 'ResNet18 with Transfer Learning',
    'Number of Classes': len(class_names),
    'Classes': list(class_names),
    'Input Size': '224x224 RGB',
    'Preprocessing': 'CLAHE Enhancement + ImageNet Normalization',
    'Best Validation Accuracy': f"{best_val_acc:.2f}%",
    'Test Accuracy': f"{accuracy_score(test_labels, test_preds)*100:.2f}%",
    'Total Parameters': f"{sum(p.numel() for p in model.parameters()):,}",
    'Training Epochs': num_epochs,
    'Batch Size': batch_size
}

print("\nModel Information:")

print("=" * 50)    print(f"{key}: {value}")
for key, value in model_info.items():

## 14. Inference Function for Deployment

In [ ]:
def load_model_for_inference(model_path, device='cpu'):
    """
    Load trained model for inference
    """
    checkpoint = torch.load(model_path, map_location=device)
    
    # Initialize model
    model = DentalClassifier(num_classes=checkpoint['num_classes'])
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    
    return model, checkpoint['class_names'], checkpoint['normalization_params']

def predict_image(model, image_path, class_names, norm_params, device='cpu'):
    """
    Predict class for a single image
    """
    # Preprocess image
    image = apply_clahe(image_path)
    if image is None:
        return None, None, None
    
    # Convert to PIL and apply transforms
    image = Image.fromarray(image)
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=norm_params['mean'], std=norm_params['std'])
    ])
    
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    # Make prediction
    with torch.no_grad():
        output = model(image_tensor)
        probabilities = F.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, 1)
    
    predicted_class = class_names[predicted.item()]
    confidence_score = confidence.item()
    all_probabilities = {class_names[i]: prob.item() for i, prob in enumerate(probabilities[0])}
    
    return predicted_class, confidence_score, all_probabilities

# Test inference function
print("Testing inference function...")
test_model, test_class_names, test_norm_params = load_model_for_inference(final_model_path, device)

# Test on a sample image
sample_image_path = df.sample(1).iloc[0]['image_path']
pred_class, confidence, all_probs = predict_image(test_model, sample_image_path, test_class_names, test_norm_params, device)

print(f"\nSample Prediction:")
print(f"Image: {os.path.basename(sample_image_path)}")
print(f"Predicted Class: {pred_class}")
print(f"Confidence: {confidence:.4f}")
print(f"All Probabilities: {all_probs}")

## 15. Summary and Next Steps

In [ ]:
print("🦷 DENTAL RADIOGRAPHY ML PIPELINE - COMPLETE! 🦷")
print("=" * 60)
print(f"✅ Dataset: {len(df)} images across {len(class_names)} classes")
print(f"✅ Model: ResNet18 with transfer learning")
print(f"✅ Preprocessing: CLAHE enhancement + data augmentation")
print(f"✅ Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"✅ Test Accuracy: {accuracy_score(test_labels, test_preds)*100:.2f}%")
print(f"✅ Model saved: {final_model_path}")
print("\n🚀 READY FOR DEPLOYMENT!")
print("\nNext Steps:")
print("1. Integrate with FastAPI backend (main.py)")
print("2. Deploy with Streamlit frontend (app.py)")
print("3. Set up Docker containers")
print("4. Configure load balancing with Nginx")
print("5. Implement monitoring and logging")
print("\n📊 Model is ready for production use!")